# Xiangqi — Colab Dataset Generator

Notebook chuyên dùng để **sinh dữ liệu các ván cờ tướng trên CPU**, với Opening Book, Alpha-Beta, checkpoint/resume và thống kê sau khi hoàn thành. Không chạy unit test hoặc benchmark trong notebook này; các bài kiểm thử nằm độc lập trong `project/backend/tests`.

Dữ liệu ghi theo định dạng **JSONL (mỗi ván một dòng)** và lưu vào Google Drive. Nếu runtime bị ngắt, chạy lại các cell với cùng `RUN_NAME` để tiếp tục các `game_index` còn thiếu. Ván đang chơi dở sẽ được tính lại từ đầu.

Dùng Colab CPU đúng giới hạn tài nguyên cho phép; thuật toán hiện tại không cần GPU.

## 1. Kiểm tra tài nguyên và tải source

In [ ]:
import os
import platform
import subprocess
import sys
from pathlib import Path

print('Python:', sys.version.split()[0])
print('CPU vCPUs:', os.cpu_count())
print('Architecture:', platform.platform())

REPO_DIR = Path('/content/ML-Advance-Group')
BRANCH = 'perf/alphabeta-batch-resume'
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH,
        'https://github.com/Ko4rut/ML-Advance-Group.git', str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)

BACKEND_DIR = REPO_DIR / 'project' / 'backend'
assert (BACKEND_DIR / 'src' / 'training' / 'batch_resume.py').exists()
print('Backend ready:', BACKEND_DIR)
print('Revision:', subprocess.check_output(
    ['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True,
).strip())

## 2. Mount Google Drive và lưu checkpoint

Dataset ghi **sau mỗi ván**, checkpoint ghi thêm `completed_indices`, `target_games` và cấu hình. Khi khởi động lại, chương trình đọc **JSONL thực tế** trước để xác định game_index đã xong; checkpoint có thể được phục hồi nếu lần trước bị ngắt giữa lúc cập nhật. Không ghi trùng seed/index.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = Path('/content/drive/MyDrive/XiangqiAI/datasets')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('Output folder:', SAVE_DIR)

## 3. Cấu hình Opening Book, Top-K exploration và khử trùng lặp

Mỗi ván bắt đầu bằng một khai cuộc có cấu trúc, sau đó Alpha-Beta tìm kiếm. Có thể bật **Top-K exploration** để chọn các nước gần tốt nhất (được tìm kiếm đủ độ sâu), giúp ván cờ đa dạng mà không random giá trị Xe/Pháo/Mã.

**Deduplicate:** trước khi ghi JSONL, hệ thống so fingerprint toàn bộ chuỗi hành động với các ván đã lưu. Nếu trùng thì sinh lại với seed khám phá khác, tối đa `MAX_DUPLICATE_RETRIES` lần; nếu vẫn không có ván mới, chương trình dừng báo lỗi chứ không ghi trùng.

Các nước `alphabeta_exploration` được đánh nhãn khác với `alphabeta`. Đổi tên `RUN_NAME` khi bật tính năng mới; **không resume vào dataset cũ**. Giữ cùng tên và cấu hình để resume batch mới.

In [ ]:
RED_AGENT = 'alphabeta'
BLACK_AGENT = 'alphabeta'
RED_DEPTH = 3
BLACK_DEPTH = 3
GAMES = 50
MAX_PLIES = 240
OPENING_MODE = 'book'    # 'book', 'random' hoặc 'none'
OPENING_PLIES = 6       # tối đa 6 ply trong curated book
NO_PROGRESS_LIMIT = 0
SEED = 42

RUN_NAME = 'alphabeta_depth3_book_topk3_dedup_v2_seed42_max240'
# Tăng độ đa dạng mà vẫn giữ nguyên PIECE_VALUES
DEDUPLICATE_GAMES = True
MAX_DUPLICATE_RETRIES = 12
EXPLORATION_TOP_K = 3
EXPLORATION_SCORE_MARGIN = 12
EXPLORATION_TEMPERATURE = 8.0
EXPLORATION_RATE = 0.35
EXPLORATION_WINDOW = 20
EXPLORATION_MAX_MOVES = 2

WORKERS = 1
PLY_PROGRESS_EVERY = 20

OUTPUT_FILE = SAVE_DIR / f'{RUN_NAME}.jsonl'
CHECKPOINT_FILE = SAVE_DIR / f'{RUN_NAME}.jsonl.checkpoint.json'
print('Opening:', OPENING_MODE, 'plies:', OPENING_PLIES)
print('Dataset:', OUTPUT_FILE)
print('Checkpoint:', CHECKPOINT_FILE)
print('Already exists:', OUTPUT_FILE.exists())

## 4. Chạy batch, theo dõi và tiếp tục

Mỗi ván bắt đầu sẽ in `[game X] started`; mỗi `PLY_PROGRESS_EVERY` lượt sẽ báo `ply/giới hạn`. Sau **mỗi ván hoàn tất**, in `[batch N/GAMES]` kèm trạng thái, thời gian ván, ETA rồi ghi JSONL và checkpoint. Có thể ngắt runtime và chạy lại; ván chưa hoàn thành sẽ chạy từ đầu. `WORKERS > 1` chạy nhiều ván độc lập song song; log ply được tắt để tránh lẫn thứ tự.

**Lưu ý:** Drive mount có thể đồng bộ trễ; không chạy đồng thời hai notebook ghi cùng RUN_NAME.

In [ ]:
command = [
    sys.executable, '-u', '-m', 'src', 'batch',
    '--red', RED_AGENT, '--black', BLACK_AGENT,
    '--red-depth', str(RED_DEPTH), '--black-depth', str(BLACK_DEPTH),
    '--games', str(GAMES), '--max-plies', str(MAX_PLIES),
    '--opening-plies', str(OPENING_PLIES),
    '--opening-mode', OPENING_MODE,
    '--no-progress-limit', str(NO_PROGRESS_LIMIT),
    '--seed', str(SEED), '--output', str(OUTPUT_FILE),
    '--progress-every', '1',
    '--ply-progress-every', str(PLY_PROGRESS_EVERY),
    '--workers', str(WORKERS),
    '--max-duplicate-retries', str(MAX_DUPLICATE_RETRIES),
    '--exploration-top-k', str(EXPLORATION_TOP_K),
    '--exploration-score-margin', str(EXPLORATION_SCORE_MARGIN),
    '--exploration-temperature', str(EXPLORATION_TEMPERATURE),
    '--exploration-rate', str(EXPLORATION_RATE),
    '--exploration-window', str(EXPLORATION_WINDOW),
    '--exploration-max-moves', str(EXPLORATION_MAX_MOVES),
    '--resume',
]
if DEDUPLICATE_GAMES:
    command.append('--deduplicate-games')
print('Starting / resuming CPU batch generation...')
subprocess.run(command, cwd=str(BACKEND_DIR), check=True)

## 5. Thống kê kết quả và độ đa dạng khai cuộc

Tổng hợp `opening_family` / `opening_id`, tỷ lệ hoàn tất, fingerprint trùng và thời gian trung bình. Hai bên có thể đi theo cùng một khai cuộc lý thuyết nhưng các thế trung cuộc do Alpha-Beta tạo ra vẫn khác nhau.

In [ ]:
import collections
import json
import statistics

records = [json.loads(line) for line in OUTPUT_FILE.read_text(encoding='utf-8').splitlines() if line.strip()]
results = collections.Counter(
    (r['outcome']['status'], r['outcome']['reason']) for r in records
)
print('Records saved:', len(records))
print('Result breakdown:', dict(results))
if records:
    print('Average ply:', round(statistics.mean(r['ply_count'] for r in records), 2))
    print('Average seconds/game:', round(statistics.mean(r['elapsed_seconds'] for r in records), 2))
    print('Unique fingerprints:', len({r['fingerprint'] for r in records}))
    print('Full results:', [(r['game_index'], r['outcome']) for r in records])
opening_families = collections.Counter(r.get('opening_family') or 'none' for r in records)
opening_ids = collections.Counter(r.get('opening_id') or 'none' for r in records)
print('Opening family breakdown:', dict(opening_families))
print('Distinct opening variations:', len(opening_ids))
print('Opening variations:', dict(opening_ids))
print('Opening-book move count:', sum(
    move['actor'] == 'opening_book'
    for record in records for move in record['moves']
))

exploration_moves = sum(
    move['actor'] == 'alphabeta_exploration'
    for record in records for move in record['moves']
)
print('Top-K sampled moves:', exploration_moves)
print('Accepted retry attempts:', sum(r.get('retry_attempt', 0) for r in records))
print('Duplicate complete games:', len(records) - len({r['fingerprint'] for r in records}))
